EDA


In [8]:
import pandas as pd

df = pd.read_csv("../data/cs-training.csv", index_col=0)
print(df.shape)
print(df.dtypes)

print(df['SeriousDlqin2yrs'].value_counts(normalize=True))

print(df.isnull().sum()/ len(df)*100)

print(df.describe())

(150000, 11)
SeriousDlqin2yrs                          int64
RevolvingUtilizationOfUnsecuredLines    float64
age                                       int64
NumberOfTime30-59DaysPastDueNotWorse      int64
DebtRatio                               float64
MonthlyIncome                           float64
NumberOfOpenCreditLinesAndLoans           int64
NumberOfTimes90DaysLate                   int64
NumberRealEstateLoansOrLines              int64
NumberOfTime60-89DaysPastDueNotWorse      int64
NumberOfDependents                      float64
dtype: object
SeriousDlqin2yrs
0    0.93316
1    0.06684
Name: proportion, dtype: float64
SeriousDlqin2yrs                         0.000000
RevolvingUtilizationOfUnsecuredLines     0.000000
age                                      0.000000
NumberOfTime30-59DaysPastDueNotWorse     0.000000
DebtRatio                                0.000000
MonthlyIncome                           19.820667
NumberOfOpenCreditLinesAndLoans          0.000000
NumberOfTimes90Days

In [9]:
print(df['NumberOfTime30-59DaysPastDueNotWorse'].value_counts().sort_index(ascending=False).head(10))

NumberOfTime30-59DaysPastDueNotWorse
98    264
96      5
13      1
12      2
11      1
10      4
9      12
8      25
7      54
6     140
Name: count, dtype: int64


In [10]:
mask = df['NumberOfTime30-59DaysPastDueNotWorse'].isin([96, 98])
print(df.loc[mask, ['NumberOfTime30-59DaysPastDueNotWorse', 
                      'NumberOfTimes90DaysLate', 
                      'NumberOfTime60-89DaysPastDueNotWorse']].head(10))

      NumberOfTime30-59DaysPastDueNotWorse  NumberOfTimes90DaysLate  \
1734                                    98                       98   
2287                                    98                       98   
3885                                    98                       98   
4418                                    98                       98   
4706                                    98                       98   
5074                                    98                       98   
6281                                    98                       98   
7033                                    98                       98   
7118                                    98                       98   
7688                                    98                       98   

      NumberOfTime60-89DaysPastDueNotWorse  
1734                                    98  
2287                                    98  
3885                                    98  
4418                                  

Data Cleaning and Feature Engineering


In [11]:
df = df[df['age'] > 0]

In [12]:
for col in ['RevolvingUtilizationOfUnsecuredLines', 'DebtRatio']:
    cap = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=cap)

In [13]:
late_cols = ['NumberOfTime30-59DaysPastDueNotWorse',
             'NumberOfTimes90DaysLate',
             'NumberOfTime60-89DaysPastDueNotWorse']

clean_medians = {col: df.loc[~df[col].isin([96, 98]), col].median() for col in late_cols}
df['deliquency_data_error'] = df[late_cols].isin([96, 98]).any(axis=1).astype(int)

for col in late_cols:
    df.loc[df[col].isin([96, 98]), col] = clean_medians[col]

In [14]:
df['income_was_missing'] = df['MonthlyIncome'].isnull().astype(int)
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())

In [15]:
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(df['NumberOfDependents'].median())

In [16]:
print(df.shape)
print(df.isnull().sum().sum())

(149999, 13)
0


In [17]:
income_cap = df['MonthlyIncome'].quantile(0.99)
df['MonthlyIncome'] = df['MonthlyIncome'].clip(upper=income_cap)

In [18]:
# Aggregation: combine the 3 late-payment columns into one total
df['TotalTimesLate'] = (df['NumberOfTime30-59DaysPastDueNotWorse'] +
                         df['NumberOfTimes90DaysLate'] +
                         df['NumberOfTime60-89DaysPastDueNotWorse'])

# Ratio: capacity (income) relative to obligation (dependents)
# +1 avoids divide-by-zero for people with 0 dependents
df['IncomePerDependent'] = df['MonthlyIncome'] / (df['NumberOfDependents'] + 1)

print(df[['TotalTimesLate', 'IncomePerDependent']].describe())

       TotalTimesLate  IncomePerDependent
count   149999.000000       149999.000000
mean         0.400349         4422.384444
std          1.101292         3180.155310
min          0.000000            0.000000
25%          0.000000         2151.000000
50%          0.000000         4000.000000
75%          0.000000         5400.000000
max         19.000000        23000.000000


In [19]:
df['TotalCreditLines'] = df['NumberOfOpenCreditLinesAndLoans'] + df['NumberRealEstateLoansOrLines']

In [20]:
print(df[['TotalCreditLines', 'IncomePerDependent']].describe())

       TotalCreditLines  IncomePerDependent
count     149999.000000       149999.000000
mean           9.471010         4422.384444
std            5.727412         3180.155310
min            0.000000            0.000000
25%            5.000000         2151.000000
50%            9.000000         4000.000000
75%           13.000000         5400.000000
max          112.000000        23000.000000


In [25]:
df.to_csv('../data/cleaned_loan_data.csv', index=False)